In [ ]:
!pip install chromadb
!pip install langchain langchain-community
!pip install langchain-chroma
!pip install -U protobuf==4.23.4
!pip install python-dotenv

https://github.com/run-llama/llama_index/tree/main/docs/examples/data/paul_graham

In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] ="sk-proj-IeiPd3N4ZP..."

In [ ]:
import os
import chromadb
from operator import itemgetter
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print(api_key)

In [ ]:
loader = TextLoader("./data/paul_graham_essay.txt", encoding="utf-8")
docs = loader.load()

splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splitted_docs = splitter.split_documents(docs)

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
vectordb = Chroma.from_documents(splitted_docs, embedding, persist_directory="./chroma_db")

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
retriever = vectordb.as_retriever()

prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트를 사용해 질문에 답하세요.\n\n{context}\n\n질문: {input}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# retrieval → prompt → llm 체인
qa_chain = (
    {
        "context": itemgetter("input") | retriever | format_docs,
        "input": itemgetter("input"),
    }
    | prompt
    | llm
)

In [ ]:
query = "What did the author do growing up?"
result = qa_chain.invoke({"input": query})
print(result.content)

In [ ]:
query = "이 문서에서 중요한 요점이 뭐야?"
result = qa_chain.invoke({"input": query})
print(result.content)